# FASE 4 — Analisis Statistik / Clustering (Hari 11–14)
**Tujuan:** Mengelompokkan kabupaten di NTT berdasarkan indikator sosial-ekonomi  
**Data:** `Master_Scaled.csv` (data ternormalisasi Min-Max, 2022–2025)
---

## 0. Import Library & Load Data

In [ ]:
import pandas as pd
import numpy as np
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Load Data ──────────────────────────────────────────────────
df = pd.read_csv('Master_Scaled.csv')

# Filter kolom tahun 2025 untuk clustering utama
kolom_2025 = [c for c in df.columns if '_2025' in c]
X_2025 = df[kolom_2025].values
kabupatens = df['Kabupaten'].tolist()

short_names = {
    'Timor Tengah Selatan': 'TTS',
    'Sumba Barat Daya': 'SBD',
    'Manggarai Timur': 'Manggarai Timur',
}
labels_short = [short_names.get(k, k) for k in kabupatens]

print(f'Jumlah Kabupaten : {len(kabupatens)}')
print(f'Jumlah Fitur 2025: {len(kolom_2025)}')
print(f'Kabupaten        : {kabupatens}')
df[['Kabupaten'] + kolom_2025].head()

---
## STEP A — Tentukan K Optimal

### Langkah 1 — Hierarchical Clustering + Dendrogram
Menggunakan `AgglomerativeClustering` dengan **Ward linkage** untuk melihat struktur hierarki data.  
Cut point terbaik diambil di jarak terpanjang (gap terbesar antar level).

In [ ]:
Z = linkage(X_2025, method='ward')

fig, ax = plt.subplots(figsize=(12, 6))
dendrogram(
    Z,
    labels=labels_short,
    ax=ax,
    color_threshold=1.5,
    above_threshold_color='#457B9D',
    leaf_rotation=45,
    leaf_font_size=10
)
ax.axhline(y=1.5, color='#E63946', linestyle='--', linewidth=1.8, label='Cut point optimal')
ax.set_title('Dendrogram — Hierarchical Clustering (Ward Linkage)\n'
             'Kabupaten NTT Berdasarkan Indikator Sosial-Ekonomi 2025',
             fontsize=13, fontweight='bold', pad=14)
ax.set_ylabel('Jarak (Euclidean Ward)', fontsize=10)
ax.set_xlabel('Kabupaten', fontsize=10)
ax.legend(fontsize=9)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('step1_dendrogram.png', dpi=150, bbox_inches='tight')
plt.show()
print('Dendrogram menunjukkan gap terpanjang → K optimal sekitar 3–4')

### Langkah 2 — Elbow Method
Menghitung **WCSS (Within-Cluster Sum of Squares)** untuk K = 2, 3, 4, 5, 6.  
Titik *siku* adalah K di mana penurunan WCSS mulai tidak signifikan.

In [ ]:
K_range = range(2, 7)
wcss = []

for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=50, random_state=42)
    km.fit(X_2025)
    wcss.append(km.inertia_)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(list(K_range), wcss, 'o-', color='#E63946',
        linewidth=2.5, markersize=9, markerfacecolor='white', markeredgewidth=2)
ax.axvline(x=4, color='#457B9D', linestyle='--', linewidth=1.5, label='K = 4 (siku)')
for k, w in zip(K_range, wcss):
    ax.annotate(f'{w:.2f}', (k, w), textcoords='offset points',
                xytext=(8, 6), fontsize=9, color='#333')
ax.set_title('Elbow Method — Penentuan K Optimal\n(Within-Cluster Sum of Squares)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Jumlah Klaster (K)', fontsize=10)
ax.set_ylabel('WCSS (Inertia)', fontsize=10)
ax.set_xticks(list(K_range))
ax.legend(fontsize=9)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('step2_elbow.png', dpi=150, bbox_inches='tight')
plt.show()

print('WCSS per K:')
for k, w in zip(K_range, wcss):
    print(f'  K={k}: {w:.4f}')

### Langkah 3 — Silhouette Score per K
Menghitung **Silhouette Score** untuk setiap nilai K.  
Pilih K dengan skor tertinggi — semakin mendekati 1 semakin baik pemisahan klaster.

In [ ]:
sil_scores = []
for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=50, random_state=42)
    labels_tmp = km.fit_predict(X_2025)
    sil_scores.append(silhouette_score(X_2025, labels_tmp))

K_optimal = list(K_range)[np.argmax(sil_scores)]

fig, ax = plt.subplots(figsize=(8, 5))
bar_colors = ['#E63946' if k == K_optimal else '#A8DADC' for k in K_range]
bars = ax.bar(list(K_range), sil_scores, color=bar_colors,
              edgecolor='white', linewidth=1.2, width=0.6)
for bar, score in zip(bars, sil_scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.004,
            f'{score:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_title(f'Silhouette Score per K (K Optimal = {K_optimal})',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Jumlah Klaster (K)', fontsize=10)
ax.set_ylabel('Silhouette Score', fontsize=10)
ax.set_xticks(list(K_range))
ax.set_ylim(0, max(sil_scores) + 0.08)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('step3_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Tabel Silhouette Score:')
sil_df = pd.DataFrame({'K': list(K_range), 'Silhouette Score': sil_scores})
sil_df['Status'] = sil_df['Silhouette Score'].apply(
    lambda x: '★ OPTIMAL' if x == max(sil_scores) else '')
print(sil_df.to_string(index=False))
print(f'\n→ K Optimal = {K_optimal}')

---
## STEP B — Jalankan K-Means

### Langkah 4 — K-Means Clustering (Tahun 2025)
Menjalankan K-Means dengan `K_optimal`, menggunakan `init='k-means++'` dan `n_init=50`  
untuk memastikan inisialisasi yang stabil dan hasil reproducible.

In [ ]:
km_final = KMeans(
    n_clusters=K_optimal,
    init='k-means++',
    n_init=50,
    random_state=42
)
df['klaster_2025'] = km_final.fit_predict(X_2025) + 1  # 1-indexed

print('Label Klaster per Kabupaten (2025):')
print(df[['Kabupaten', 'klaster_2025']].to_string(index=False))
print(f'\nDistribusi Klaster:')
print(df['klaster_2025'].value_counts().sort_index().to_string())

### Langkah 5 — Clustering per Tahun (2022, 2023, 2024)
Menjalankan K-Means terpisah untuk setiap tahun dengan K yang sama.  
Hasilnya disimpan ke kolom `klaster_2022`, `klaster_2023`, `klaster_2024`.

In [ ]:
for tahun in [2022, 2023, 2024]:
    kolom_t = [c for c in df.columns if f'_{tahun}' in c]
    X_t = df[kolom_t].values
    km_t = KMeans(n_clusters=K_optimal, init='k-means++', n_init=50, random_state=42)
    df[f'klaster_{tahun}'] = km_t.fit_predict(X_t) + 1

print('Klaster per Tahun:')
print(df[['Kabupaten','klaster_2022','klaster_2023','klaster_2024','klaster_2025']].to_string(index=False))

### Langkah 6 — Label Klaster Sesuai Profil
Menghitung rata-rata tiap variabel per klaster, kemudian memberi label manual:  
- **Rawan** → kemiskinan + stunting tertinggi  
- **Berkembang** → kondisi menengah  
- **Sejahtera** → kemiskinan + stunting terendah

In [ ]:
kolom_label = [
    'Persentase Penduduk Miskin (Persen)_2025',
    'Jumlah Balita Stunting (Jiwa)_2025',
    'Total Rata-rata Upah/Gaji Bersih (Rupiah)_2025',
    'PRDB per Kapita AHB (Ribu Rupiah)_2025',
    'Akses Air Minum Layak (Persen)_2025',
    'Akses Sanitasi Layak (Persen)_2025'
]
profil = df.groupby('klaster_2025')[kolom_label].mean()
skor_risiko = profil['Persentase Penduduk Miskin (Persen)_2025'] + \
              profil['Jumlah Balita Stunting (Jiwa)_2025']

sorted_klaster = skor_risiko.sort_values(ascending=False).index.tolist()
label_map = {}
for i, k in enumerate(sorted_klaster):
    if i == 0: label_map[k] = 'Rawan'
    elif i == len(sorted_klaster)-1: label_map[k] = 'Sejahtera'
    else: label_map[k] = f'Berkembang {i}'

df['label_klaster'] = df['klaster_2025'].map(label_map)

# Heatmap profil
profil_display = profil.copy()
profil_display.index = [f"K{i}—{label_map.get(i,'')}" for i in profil_display.index]
profil_display.columns = ['%Miskin','Stunting','Upah','PDRB/kap','AirMinum','Sanitasi']

fig, ax = plt.subplots(figsize=(11, 4))
sns.heatmap(profil_display, annot=True, fmt='.3f', cmap='RdYlGn_r',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Nilai Ternormalisasi (0-1)'})
ax.set_title('Profil Rata-rata Tiap Klaster — Indikator Utama 2025',
             fontsize=12, fontweight='bold', pad=12)
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('step6_profil_klaster.png', dpi=150, bbox_inches='tight')
plt.show()

print('Label Klaster:')
print(df[['Kabupaten','klaster_2025','label_klaster']].to_string(index=False))

---
## STEP C — Validasi

### Langkah 7 — Silhouette Index Final
Menghitung Silhouette Score untuk hasil K-Means final.  
> Nilai > 0.5 = Baik | > 0.7 = Sangat Baik

In [ ]:
X_scaled = X_2025
labels_final = km_final.labels_

sil_final = silhouette_score(X_scaled, labels_final)
print(f'Silhouette Index Final : {sil_final:.4f}')
if sil_final > 0.7: kualitas = 'Sangat Baik'
elif sil_final > 0.5: kualitas = 'Baik'
elif sil_final > 0.25: kualitas = 'Cukup (struktur klaster ada namun lemah)'
else: kualitas = 'Lemah'
print(f'Interpretasi           : {kualitas}')

### Langkah 8 — Davies-Bouldin Index
Nilai mendekati **0** = pemisahan klaster sangat baik.

In [ ]:
db_final = davies_bouldin_score(X_scaled, labels_final)
print(f'Davies-Bouldin Index: {db_final:.4f}')
print(f'Interpretasi        : Semakin mendekati 0 semakin baik pemisahan klaster')

print(f'\n=== Ringkasan Validasi ===')
print(f'K Optimal        : {K_optimal}')
print(f'Silhouette Score : {sil_final:.4f} — {kualitas}')
print(f'Davies-Bouldin   : {db_final:.4f}')

### Langkah 9 — Visualisasi 2D dengan PCA
Mereduksi dimensi ke 2D menggunakan PCA untuk visualisasi scatter plot klaster.  
Gambar ini digunakan dalam essay untuk menunjukkan pemisahan klaster.

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
var_explained = pca.explained_variance_ratio_ * 100

label_names = df['label_klaster'].values
color_map = {'Rawan': '#E63946', 'Berkembang 1': '#F4A261',
             'Berkembang 2': '#E9C46A', 'Sejahtera': '#2A9D8F'}
marker_map = {'Rawan': 'X', 'Berkembang 1': 'D',
              'Berkembang 2': 's', 'Sejahtera': 'o'}

fig, ax = plt.subplots(figsize=(10, 7))
for nama_label in color_map:
    mask = label_names == nama_label
    if mask.sum() == 0: continue
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=color_map[nama_label], marker=marker_map[nama_label],
               s=180, label=nama_label, edgecolors='white', lw=1.2, zorder=5)
    for i, kab in enumerate(np.array(kabupatens)[mask]):
        ax.annotate(short_names.get(kab, kab),
                    (X_pca[mask][i, 0], X_pca[mask][i, 1]),
                    xytext=(7, 4), textcoords='offset points', fontsize=8.5)

ax.set_xlabel(f'PC1 ({var_explained[0]:.1f}% variansi)', fontsize=10)
ax.set_ylabel(f'PC2 ({var_explained[1]:.1f}% variansi)', fontsize=10)
ax.set_title(f'Scatter Plot 2D — PCA Visualisasi K-Means (K={K_optimal})\n'
             f'Silhouette={sil_final:.3f} | Davies-Bouldin={db_final:.3f}',
             fontsize=12, fontweight='bold', pad=12)
ax.legend(title='Klaster', fontsize=9, title_fontsize=9, framealpha=0.9)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('step9_pca_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Variansi yang dijelaskan PC1+PC2: {sum(var_explained):.1f}%')

---
## STEP D — Analisis Lanjutan

### Langkah 10 — Analisis PCA: Kontribusi Variabel
Melihat `pca.components_` untuk mengetahui variabel mana yang paling berkontribusi  
ke **PC1** dan **PC2** — ini menjadi argumen 'harga beras berkontribusi signifikan'.

In [ ]:
loadings = pd.DataFrame(
    pca.components_.T,
    index=[c.replace('_2025', '') for c in kolom_2025],
    columns=['PC1', 'PC2']
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, pc in enumerate(['PC1', 'PC2']):
    sorted_load = loadings[pc].sort_values()
    colors = ['#E63946' if v < 0 else '#2A9D8F' for v in sorted_load]
    axes[i].barh(range(len(sorted_load)), sorted_load.values,
                 color=colors, edgecolor='white')
    axes[i].set_yticks(range(len(sorted_load)))
    axes[i].set_yticklabels([l[:30] for l in sorted_load.index], fontsize=8.5)
    axes[i].axvline(0, color='#333', lw=0.8)
    axes[i].set_title(f'{pc} ({pca.explained_variance_ratio_[i]*100:.1f}% variansi)\nKontribusi Variabel',
                      fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Loading Score')
    axes[i].spines[['top','right']].set_visible(False)
plt.suptitle('Analisis PCA — Kontribusi Variabel ke PC1 & PC2',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('step10_pca_loadings.png', dpi=150, bbox_inches='tight')
plt.show()

print('Tabel Loading PCA (diurutkan berdasarkan PC1):')
print(loadings.sort_values('PC1', ascending=False).round(4).to_string())

### Langkah 11 — Analisis Transisi Klaster
Membuat crosstab matriks transisi: baris = klaster asal, kolom = klaster tujuan.  
Menghitung berapa kabupaten yang berpindah klaster antar tahun.

In [ ]:
trans_2022_2023 = pd.crosstab(df['klaster_2022'], df['klaster_2023'],
                               rownames=['Klaster 2022'], colnames=['Klaster 2023'])
trans_2023_2024 = pd.crosstab(df['klaster_2023'], df['klaster_2024'],
                               rownames=['Klaster 2023'], colnames=['Klaster 2024'])
trans_2024_2025 = pd.crosstab(df['klaster_2024'], df['klaster_2025'],
                               rownames=['Klaster 2024'], colnames=['Klaster 2025'])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax_i, (trans, title) in enumerate(zip(
        [trans_2022_2023, trans_2023_2024, trans_2024_2025],
        ['2022 → 2023', '2023 → 2024', '2024 → 2025'])):
    all_k = sorted(set(trans.index.tolist() + trans.columns.tolist()))
    trans_r = trans.reindex(index=all_k, columns=all_k, fill_value=0)
    sns.heatmap(trans_r, annot=True, fmt='d', cmap='Blues',
                ax=axes[ax_i], linewidths=0.5, cbar=False, vmin=0)
    axes[ax_i].set_title(f'Transisi Klaster {title}', fontsize=11, fontweight='bold')
    axes[ax_i].set_xlabel('Klaster Tujuan')
    axes[ax_i].set_ylabel('Klaster Asal')
plt.suptitle('Matriks Transisi Klaster Antar Tahun', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('step11_transisi.png', dpi=150, bbox_inches='tight')
plt.show()

print('Transisi 2022 → 2023:\n', trans_2022_2023)
print('\nTransisi 2023 → 2024:\n', trans_2023_2024)
print('\nTransisi 2024 → 2025:\n', trans_2024_2025)

### Langkah 12 — Identifikasi Early Warning
Flag kabupaten yang pindah ke klaster lebih buruk **2 tahun berturut-turut**:  
- `2022→2023` risiko naik **DAN** `2023→2024` risiko naik → masuk daftar prioritas  
- `2023→2024` risiko naik **DAN** `2024→2025` risiko naik → early warning terbaru

In [ ]:
# Hitung skor risiko per klaster per tahun
def get_risiko_klaster(klaster_col, tahun):
    kolom_t = [c for c in df.columns if f'_{tahun}' in c and
               ('Penduduk Miskin' in c or 'Stunting' in c)]
    if not kolom_t: return {k: k for k in df[klaster_col].unique()}
    return df.groupby(klaster_col)[kolom_t].mean().mean(axis=1)

risiko = {}
for tahun in [2022, 2023, 2024, 2025]:
    risiko[tahun] = get_risiko_klaster(f'klaster_{tahun}', tahun)
    df[f'risiko_{tahun}'] = df[f'klaster_{tahun}'].map(risiko[tahun])

# Flag transisi memburuk
df['ew_2022_2023'] = df['risiko_2023'] > df['risiko_2022']
df['ew_2023_2024'] = df['risiko_2024'] > df['risiko_2023']
df['ew_2024_2025'] = df['risiko_2025'] > df['risiko_2024']

df['early_warning']         = df['ew_2022_2023'] & df['ew_2023_2024']  # 2022-2024
df['early_warning_terbaru'] = df['ew_2023_2024'] & df['ew_2024_2025']  # 2023-2025

# Tampilkan daftar prioritas
ew_cols = ['Kabupaten','klaster_2022','klaster_2023','klaster_2024','klaster_2025',
           'label_klaster','early_warning','early_warning_terbaru']
ew_all = df[df['early_warning'] | df['early_warning_terbaru']][ew_cols].copy()
ew_all['early_warning'] = ew_all['early_warning'].map({True:'⚠ YA', False:'—'})
ew_all['early_warning_terbaru'] = ew_all['early_warning_terbaru'].map({True:'⚠ YA', False:'—'})

print('=== Daftar Kabupaten Early Warning ===')
print(ew_all.to_string(index=False))
print(f'\nJumlah Kabupaten Early Warning (2022–2024) : {(df["early_warning"]).sum()}')
print(f'Jumlah Kabupaten Early Warning (2023–2025) : {(df["early_warning_terbaru"]).sum()}')

---
## Simpan Hasil & Ringkasan Akhir

In [ ]:
# Simpan hasil lengkap
df.to_csv('Hasil_Clustering_NTT.csv', index=False)
print('✓ File disimpan: Hasil_Clustering_NTT.csv')

# Ringkasan
print('\n' + '='*50)
print('RINGKASAN HASIL FASE 4 — CLUSTERING NTT')
print('='*50)
print(f'K Optimal (Silhouette)  : {K_optimal}')
print(f'Silhouette Index Final  : {sil_final:.4f} — {kualitas}')
print(f'Davies-Bouldin Index    : {db_final:.4f}')
print(f'Variansi PC1+PC2        : {sum(var_explained):.1f}%')
print()
print('Distribusi Klaster 2025:')
for k, label in sorted(label_map.items()):
    kab_list = df[df['klaster_2025']==k]['Kabupaten'].tolist()
    print(f'  K{k} [{label:15s}]: {kab_list}')
print()
ew_kabupaten = df[df['early_warning_terbaru']]['Kabupaten'].tolist()
print(f'Early Warning Terbaru (2023→2025): {ew_kabupaten}')